## 01 Preprocess Tarif Area

In [1]:
# imports and inread data
import geopandas as gpd

tarif_data = gpd.read_file('../data/tarifverbundkarte/tarifverbundkarte.shp')
tarif_data.head()

,id,source,level,name,partners,partnercode,tarifverbun,partners_js,geometry
0,2296,verbund,1,Ascona,Arcobaleno,464,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((8.7766 46.16552, 8.79613 46.14906, 8..."
1,2301,verbund,1,Aubonne,Mobilis,450,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((6.3587 46.51891, 6.37025 46.51034, 6..."
2,2318,verbund,1,Bachs,ZVV,490,http://www.sbb.ch/abos-billette/tarifverbunde/...,"{""url"": ""http://www.sbb.ch/abos-billette/tarif...","POLYGON ((8.4175 47.54569, 8.42227 47.54588, 8..."
3,2322,verbund,1,Balgach,Ostwind,452,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((9.62464 47.41284, 9.62507 47.40911, ..."
4,2324,verbund,1,Ballens,Mobilis,450,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((6.4005 46.53979, 6.39567 46.53989, 6..."


In [2]:
# Merge municipalities by single partner (multi-partner municipalities are used for each partner)
tarif_for_merge = tarif_data[['partners', 'geometry']].copy()
tarif_for_merge = tarif_for_merge.dropna(subset=['partners']).copy()

tarif_for_merge['partner_list'] = (
    tarif_for_merge['partners']
    .astype(str)
    .str.split(',')
    .apply(lambda values: [value.strip() for value in values if value and value.strip()])
)

# Duplicate rows for municipalities that have multiple partners
tarif_by_partner = tarif_for_merge.explode('partner_list').rename(columns={'partner_list': 'partner'})
tarif_by_partner = tarif_by_partner[tarif_by_partner['partner'].notna() & (tarif_by_partner['partner'] != '')].copy()

# Dissolve to one polygon per partner
merged_by_partner = tarif_by_partner.dissolve(by='partner', as_index=False, aggfunc='first')

# Export merged polygons to GeoJSON
merged_by_partner[['partner', 'geometry']].to_file(
    '../data/tarifverbundkarte/tarifverbund_merged_by_partner.geojson',
    driver='GeoJSON'
 )

merged_by_partner[['partner', 'geometry']].head()

/opt/anaconda3/envs/gda/lib/python3.13/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


,partner,geometry
0,A-Welle,"MULTIPOLYGON (((7.67797 47.2477, 7.6727 47.245..."
1,Arcobaleno,"MULTIPOLYGON (((8.85909 45.96704, 8.85908 45.9..."
2,Frimobil,"POLYGON ((6.83806 46.49942, 6.83274 46.50507, ..."
3,Libero,"MULTIPOLYGON (((7.19542 46.89898, 7.18771 46.9..."
4,Léman Pass,"POLYGON ((5.97774 46.13376, 5.97635 46.13263, ..."


In [2]:
# chosen_tarifverbund = "Tarifverbund Schwyz"
chosen_tarifverbund = "Vagabond"

In [3]:
# filter for entries containing the specified tarif union
filtered_data = tarif_data[tarif_data['partners'].str.contains(chosen_tarifverbund, na=False)].copy()

filtered_data.head()

,id,source,level,name,partners,partnercode,tarifverbun,partners_js,geometry
26,2435,verbund,1,Boncourt,Vagabond,479,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((7.03625 47.49901, 7.03697 47.4975, 7..."
68,2641,verbund,1,Courrendlin,Vagabond,479,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((7.36369 47.35592, 7.36711 47.35249, ..."
76,2690,verbund,1,Develier,Vagabond,479,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((7.31458 47.35959, 7.30833 47.35176, ..."
94,2853,verbund,1,Fontenais,Vagabond,479,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((7.0972 47.40274, 7.09136 47.39366, 7..."
156,3169,verbund,1,La Baroche,Vagabond,479,https://www.sbb.ch/de/abos-billette/kaufmoegli...,"{""url"": ""https://www.sbb.ch/de/abos-billette/k...","POLYGON ((7.18311 47.44119, 7.18882 47.43967, ..."


In [4]:
# merge into one single geometry
merged_geometry = filtered_data.union_all()
merged_gdf = gpd.GeoDataFrame(geometry=[merged_geometry], crs=tarif_data.crs)

# export as a shapefile
# merged_gdf.to_file('../data/tarif-area/tarifverbund_schwyz_merged.shp')
merged_gdf.to_file('../data/tarif-area/vagabond_merged.shp')